# Transfermarkt data cleaning part 2

### Introduction
This notebook is mainly about engineering new features and making final preparation of the Transfermarkt data.

### Feature engineering
The aim is to create features realted to: **player influence, match importance and team presure**.All of these feature will be extremely important for the models and their predictions and all of them require a very detail and strict preprocesing which includes almost all of the transfermarkt datasets.

### About the features:
#### Players influence:
The impact of the football players during the matches is one of the most important things which influences the outcomes of the matches and their possible predictions.In order to create such kind of features I should consider many different factors such as: minutes played, goals, assists, injuries etc.I will identify if a player is a **key player**, which would mean a player which is playing constantly and also a **star player** which refers to the team most effective and outstanding player!

##### Possible features to be created:
- missing_players_count
- missing_key_players_count
- missing_star_players_count
- missing_importance_sum - The sum of the missing players importance score
- missing_importance_ratio - The division of the missing players importance score with the total team squad importance
- missing_defenders
- missing_midfielders
- missing_forwards
- starting_goalkeeper_missing

The features are more or less self-explanatory. \
These are the ones which are comming to my mind at the moment but for sure there will be more.

#### Match importance:
The matches themselves are not equally important!And this is the thing which I want to capture and identify!In order to do so, I need to consider other competitions besides LaLiga and this is the reason why I have collected so many data about so many competitions and matches.Alongside with the competitions, there are also factors such as fighting for a title or to not be dropped out of the division which is a problem for the last three or four teams in the league!There also more things which should be taken into account but this will happen as the work progress!

I want to specify that the match importance is directly related to the team presure, so keep this in mind!

##### Possible features to be created:
### Match Importance
- distance_to_title - counted as points gathered
- distance_to_top4
- distance_to_relegation - The team to drop out of the **first division**(which is LaLiga) and to go into the **second**!
- title_race_flag - If the team is fighting for the title
- relegation_battle_flag - If the match is between teams which both are fighting for their places into the division
- europe_race_flag - If the team is participating in another european competition
- days_until_next_match
- next_match_priority - **The matches should be ranked by priority(e.g World Cup: 1; Champions League: 2; Copa Del Ray: 3; La Liga: 4)**
- important_match_ahead - this includes matches for more important competitions in which the specific teams is participating in!
- matches_last_14_days
- rotation_risk

> It is important to mention that all of these features will be calculated and processed at the moment of the matches and there will not be introduced any data leakage, especially for the players influence features.

### Preprocessing
Before creating the actual features I need to deal with the `appearances`, `player_injuries` and `lineups` datasets.Actually, from these datasets the main and most important information will be collected.

#### About the appearances dataset:
The dataset contains the appearances of all players of all clubs in all of the competitions.This is the dataset from which the player's importance scores will be calculated.This will happen by calculating different rolling stats such as goals, assists, minuted played etc. up until the dates of the matches.

#### About the player_injuries dataset:
The dataset contains data about the injuries of each player within the dataset.The injuries are represented with their **start date** and their **end data**, including the id of the player and the type of his injury!This dataset will be used to create player's availability features!This means that for each of the teams matches in LaLiga, there will be identified if the team's players(all of them, including: **substitutes, starters, injured, suspended**) have been available at the moment of the matches!

#### About the lineups dataset:
The lineup dataset is about the line ups of the different teams for the matches.The dataset contains info about each of the line ups players for each of the matches for each of the teams within the dataset.This dataset will be used exactly to check which playres were available and which not!This is the base of the features which will be created!

#### Initial data cleaning:
In order to create the wanted features I will first need to deal with the datasets(specified above) by reducing their data and getting only that which is related to the objects of the project(The LaLiga teams and playres) and also make some surface cleaning of the datasets if it is required!

### Getting started
I will start by exploring and cleaning the `appearances, lineups` and `player_injuries` datasets:

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from football_betting_analysis.features.features_creation import create_season_feature, calculate_rolling_metric, calculate_ewma_average, calculate_time_based_ewma

from football_betting_analysis.config import DATE_FORMAT, START_DATE, END_DATE, WINDOW
from football_betting_analysis.data.data_cleaning import optimize_dataframe_memory, validate_and_cast_dataframe_dtypes

### Loading the data:
Lets load all of the data that we will be working to:

In [2]:
matches_interim_df = pd.read_parquet('../../data/interim/transfermarkt_data/interim_matches_v1.parquet')

In [3]:
competitions_df = pd.read_csv('../../data/raw/transfermarkt_data/competitions.csv')

In [4]:
appearances_df = pd.read_csv('../../data/raw/transfermarkt_data/appearances.csv')

In [5]:
lineups_df = pd.read_csv('../../data/raw/transfermarkt_data/game_lineups.csv')

C:\Users\Asus\AppData\Local\Temp\ipykernel_15892\1064156889.py:1: DtypeWarning: Columns (0: number) have mixed types. Specify dtype option on import or set low_memory=False.
  lineups_df = pd.read_csv('../../data/raw/transfermarkt_data/game_lineups.csv')


In [6]:
injuries_df = pd.read_csv('../../data/raw/transfermarkt_data/player_injuries.csv')

In [7]:
appearances_df.columns

Index(['appearance_id', 'game_id', 'player_id', 'player_club_id',
       'player_current_club_id', 'date', 'player_name', 'competition_id',
       'yellow_cards', 'red_cards', 'goals', 'assists', 'minutes_played'],
      dtype='str')

In [8]:
lineups_df.columns

Index(['game_lineups_id', 'date', 'game_id', 'player_id', 'club_id',
       'player_name', 'type', 'position', 'number', 'team_captain'],
      dtype='str')

In [9]:
injuries_df.columns

Index(['player_id', 'season_name', 'injury_reason', 'from_date', 'end_date',
       'days_missed', 'games_missed'],
      dtype='str')

In [10]:
appearances_df.shape, lineups_df.shape, injuries_df.shape

((1885697, 13), (3049833, 10), (143195, 7))

Now as we can see the datasets are really big!So I think that the first think that should be done is to reduce their size by getting only the data which is related to the objects of the project.This would mean that I should remove all of the players which have not been part from the Spanish LaLiga between the seasons **2014/2015** and **2024/2025**!This will reduce the size of the dataset **significantly**!

> I want to specify that all of these datasets are about the `players`, their appearances in different matches, their injuries and also their availability in their teams matches lineups!Just to be more clear with what data we are dealing with!

### Reducing the size's of the datasets:
How we reduce the data will then determine how the features will be constructed and also how useful and accurate they will be at all!So this is something very important and should not be done without thinking of the possible **consequences**!For example one wrong approach would be to get only the data which is from the LaLiga as a competition(`competition_id == ES1`).If we do that we will lost more of the important information that we have about the players and their appearances(in Champions league or Copa Del Ray for example) and then this will directly affect the `player influence` features!So in order to not make such mistakes I have decided to go with the following approach: I will only take the id's of the La Liga teams, and then I will filter the data by ensuring that there will not be remained competitions other than LaLiga and the other European cups(Champions League, League Europe, Copa Del Ray etc.).This is the most rightful approach because if I choose to get the id's of the players which have been a part of any season in LaLiga, this will cause problems because this way:

I do not excude the fact that these players could also have been a part to another competition in the date range we have defined.For example, **Jude Belingam**, a current Real Madrid player, has been a part of **Dortmund** before going for Real Madrid which will lead to the occurance of German competitions appearances.However, we don't want that!**We want to keep only the matches(appearances) played while the player belongs to a La Liga club**!But why this is the right approach here?Because when calculating players importances and defining the influence of a certain player, we want to do that using only tha stats of the player from the team he is currently in(which is a team from LaLiga), without carring about any other teams or competitions that he has been playing in!We should ask: **How important is the player to his current team? not: How important was he three years ago in another league?**

Enough talking, lets begin by getting the id's of the LaLiga teams.I can do that using the cleabed `matches` dataset.But before doing so, I will first need to limit the date ranges of these datasets, so that we are working only with data from the seasons: **2014/2015 and 2024/2025**!

In [11]:
appearances_df['date'] = pd.to_datetime(appearances_df['date'], format=DATE_FORMAT)
lineups_df['date'] = pd.to_datetime(lineups_df['date'], format=DATE_FORMAT)
injuries_df['from_date'] = pd.to_datetime(injuries_df['from_date'], format=DATE_FORMAT)
injuries_df['end_date'] = pd.to_datetime(injuries_df['end_date'], format=DATE_FORMAT)

In [12]:
appearances_interim_df = appearances_df[
    (appearances_df.date >= START_DATE) &
    (appearances_df.date <= END_DATE)
]

In [13]:
lineups_interim_df = lineups_df[
    (lineups_df.date >= START_DATE) &
    (lineups_df.date <= END_DATE)
]

In [14]:
injuries_interim_df = injuries_df[
    (injuries_df.from_date.between(START_DATE, END_DATE)) &
    (injuries_df.end_date.between(START_DATE, END_DATE))
]

In [15]:
appearances_interim_df.shape, lineups_interim_df.shape, injuries_interim_df.shape

((1472397, 13), (2519518, 10), (118127, 7))

In [16]:
del appearances_df, lineups_df, injuries_df

Ok, now as we limited our data into the project's defined date range, now lets get the target teams id's:

In [17]:
target_clubs = set(
    matches_interim_df.loc[
        matches_interim_df["competition_id"] == "ES1",
        "home_club_id"
    ]
).union(
    set(
        matches_interim_df.loc[
            matches_interim_df["competition_id"] == "ES1",
            "away_club_id"
        ]
    )
)

#### Reducing the size of the appearances data:

I will filter the data using the target teams id's and the player_club_id feature:

In [18]:
appearances_interim_df = appearances_interim_df[
    appearances_interim_df["player_club_id"].isin(target_clubs)
]

In [19]:
appearances_interim_df.shape

(146708, 13)

Now as an proof that I have not taken only data from the **LaLiga** competition, but the data also contains the other very important european competitions, here are all of the competitions that **Vinícius Júnior** has been participated in!This is based on the transfermarkt data and the players **of course**!

In [20]:
appearances_interim_df[appearances_interim_df.player_id == 371998].competition_id.unique()

<ArrowStringArray>
['ES1', 'CDR', 'CL', 'KLUB', 'SUC', 'USC']
Length: 6, dtype: str

#### Reducing the size of the lineups data:

In [21]:
lineups_interim_df = lineups_interim_df[
    lineups_interim_df["club_id"].isin(target_clubs)
]

In [22]:
lineups_interim_df.shape

(204082, 10)

#### Reducing the size of the players injuries data:

Now this dataset does not have club identifier, so I will just get the unique players id's from the lineups dataset and use them to filter the injuries dataset.I will use the lineups because this dataset contain more players than the appearances dataset: 

In [23]:
len(appearances_interim_df.player_id.unique()), len(lineups_interim_df.player_id.unique())

(2361, 3606)

If I use the `appearances` dataset, I will miss some players.And of course we do not want that!

In [24]:
target_players = set(
    lineups_interim_df.player_id.unique()
)

In [25]:
injuries_interim_df = injuries_interim_df[
    injuries_interim_df["player_id"].isin(target_players)
]

In [26]:
injuries_interim_df.shape

(12362, 7)

I think that overall, this what I could have done as a reduction and now it's time to observe the three datasets and make some validations to ensure that everything is perfect:

#### Exploring and Validating the `appearances, lineups and injuries` datasets:

In [27]:
appearances_interim_df.info()

<class 'pandas.DataFrame'>
Index: 146708 entries, 265258 to 1736122
Data columns (total 13 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   appearance_id           146708 non-null  str           
 1   game_id                 146708 non-null  int64         
 2   player_id               146708 non-null  int64         
 3   player_club_id          146708 non-null  int64         
 4   player_current_club_id  146708 non-null  int64         
 5   date                    146708 non-null  datetime64[us]
 6   player_name             146708 non-null  str           
 7   competition_id          146708 non-null  str           
 8   yellow_cards            146708 non-null  int64         
 9   red_cards               146708 non-null  int64         
 10  goals                   146708 non-null  int64         
 11  assists                 146708 non-null  int64         
 12  minutes_played          146708 non-null 

In [28]:
lineups_interim_df.info()

<class 'pandas.DataFrame'>
Index: 204082 entries, 256141 to 2774510
Data columns (total 10 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   game_lineups_id  204082 non-null  str           
 1   date             204082 non-null  datetime64[us]
 2   game_id          204082 non-null  int64         
 3   player_id        204082 non-null  int64         
 4   club_id          204082 non-null  int64         
 5   player_name      204082 non-null  str           
 6   type             204082 non-null  str           
 7   position         204082 non-null  str           
 8   number           204082 non-null  object        
 9   team_captain     204082 non-null  int64         
dtypes: datetime64[us](1), int64(4), object(1), str(4)
memory usage: 31.0+ MB


In [29]:
injuries_interim_df.info()

<class 'pandas.DataFrame'>
Index: 12362 entries, 47 to 143188
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   player_id      12362 non-null  int64         
 1   season_name    12362 non-null  str           
 2   injury_reason  12362 non-null  str           
 3   from_date      12362 non-null  datetime64[us]
 4   end_date       12362 non-null  datetime64[us]
 5   days_missed    12362 non-null  float64       
 6   games_missed   12362 non-null  int64         
dtypes: datetime64[us](2), float64(1), int64(2), str(2)
memory usage: 999.2 KB


There are no missing values, which is very good becase if there were any missing values we would had have problems with the calculation of the features!

Now as a minimum cleaning I will lower-cast their data types and optimize the datasets memory usage:
### Fixing the data types:

In [30]:
appearances_interim_df = validate_and_cast_dataframe_dtypes(appearances_interim_df)
lineups_interim_df = validate_and_cast_dataframe_dtypes(lineups_interim_df)
injuries_interim_df = validate_and_cast_dataframe_dtypes(injuries_interim_df)

### Optimizing the datasets memory usage:

In [31]:
appearances_interim_df = optimize_dataframe_memory(appearances_interim_df)
lineups_interim_df = optimize_dataframe_memory(lineups_interim_df)
injuries_interim_df = optimize_dataframe_memory(injuries_interim_df)

Initial Memory Usage: 19.87 MB
Final Memory Usage: 8.55 MB
Memory Reduction: 56.96%

Optimized Data Types:
appearance_id                        str
game_id                            int32
player_id                          int32
player_club_id                     int16
player_current_club_id             int32
date                      datetime64[us]
player_name                     category
competition_id                  category
yellow_cards                        int8
red_cards                           int8
goals                               int8
assists                             int8
minutes_played                     int16
dtype: object
Initial Memory Usage: 39.32 MB
Final Memory Usage: 14.12 MB
Memory Reduction: 64.10%

Optimized Data Types:
game_lineups_id               str
date               datetime64[us]
game_id                     int32
player_id                   int32
club_id                     int16
player_name              category
type                     category


Now I want to make some validations:

#### Only valid competitions: 
First, are there any other competitions than the: 
**Spanish specific**:
- LaLiga
- Copa Del Ray
- Supercopa

**Europe specific**:
- Champions League
- Europe League
- Conference League
- UEFA Super Cup - which is played between the winners of the Champions league and the Europe League

**World wide(FIFA)**:
- World Cup
- European cup

In [32]:
target_competitions = appearances_interim_df.competition_id.unique()

In [33]:
competitions_df[competitions_df.competition_id.isin(target_competitions)]

,competition_id,competition_code,name,sub_type,type,country_id,country_name,domestic_league_code,confederation,total_clubs,url
9,CDR,copa-del-rey,copa-del-rey,domestic_cup,domestic_cup,157,Spain,ES1,europa,20.0,https://www.transfermarkt.co.uk/copa-del-rey/s...
12,CL,uefa-champions-league,uefa-champions-league,uefa_champions_league,international_cup,-1,NaN,NaN,europa,NaN,https://www.transfermarkt.co.uk/uefa-champions...
13,CLQ,uefa-champions-league-qualifying,uefa-champions-league-qualifying,uefa_champions_league_qualifying,international_cup,-1,NaN,NaN,europa,NaN,https://www.transfermarkt.co.uk/uefa-champions...
20,ECLQ,uefa-conference-league-qualifiers,uefa-conference-league-qualifiers,uefa_conference_league_qualifiers,other,-1,NaN,NaN,europa,NaN,https://www.transfermarkt.co.uk/uefa-conferenc...
21,EL,uefa-europa-league,uefa-europa-league,uefa_europa_league,other,-1,NaN,NaN,europa,NaN,https://www.transfermarkt.co.uk/uefa-europa-le...
22,ELQ,uefa-europa-league-qualifying,uefa-europa-league-qualifying,uefa_europa_league_qualifying,other,-1,NaN,NaN,europa,NaN,https://www.transfermarkt.co.uk/uefa-europa-le...
23,ES1,laliga,laliga,first_tier,domestic_league,157,Spain,ES1,europa,20.0,https://www.transfermarkt.co.uk/laliga/startse...
35,KLUB,fifa-klub-wm,fifa-klub-wm,fifa_club_world_cup,other,-1,NaN,NaN,europa,NaN,https://www.transfermarkt.co.uk/fifa-klub-wm/s...
59,SUC,supercopa,supercopa,domestic_super_cup,other,157,Spain,ES1,europa,20.0,https://www.transfermarkt.co.uk/supercopa/star...
66,USC,uefa-super-cup,uefa-super-cup,uefa_super_cup,international_cup,-1,NaN,NaN,europa,NaN,https://www.transfermarkt.co.uk/uefa-super-cup...


Well from what I can see everything seems to be perfect!

#### 
Now lets ensure that the data is within the project defined date range: 2014/2015 - 2024/2025:

In [34]:
appearances_interim_df.date.dt.year.unique()

array([2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024,
       2025], dtype=int32)

In [35]:
lineups_interim_df.date.dt.year.unique()

array([2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024,
       2025], dtype=int32)

In [36]:
injuries_interim_df.season_name.unique().tolist()

['24/25',
 '23/24',
 '20/21',
 '19/20',
 '18/19',
 '17/18',
 '22/23',
 '21/22',
 '15/16',
 '16/17',
 '14/15',
 '13/14']

Ok the dates seems to be fine.

Lets check if there is an player injury with end date smaller than or equal to the start date!

In [37]:
injuries_interim_df[
    injuries_interim_df.end_date <= injuries_interim_df.from_date
].shape

(46, 7)

In [38]:
injuries_interim_df[
    injuries_interim_df.end_date == injuries_interim_df.from_date
].shape

(46, 7)

Ok there are not invaid dates such as with end date before start date, but there are one day injuries.Actually these injuries are totally valid acceptable.However, what I am more intereset in, is the fact that there are injuries which have not affected the player's matches, **which means even that certain player has been injured, he has not missed any matches**!

By the way, lets see how many are these players:

In [39]:
injuries_interim_df[injuries_interim_df.games_missed == 0].shape

(537, 7)

There are not very much but still there are some!The question is: Are these injuries useful for the features that we want to create or not?Should I remove them?Well if we look at the feature we want to create:
- missing_players_count
- missing_key_players_count
- missing_star_players_count
- missing_importance_sum - The sum of the missing players importance score
- missing_importance_ratio - The division of the missing players importance score with the total team squad importance
- missing_defenders
- missing_midfielders
- missing_forwards
- starting_goalkeeper_missing

As we can see they are all related to counts.And in order, these counts to mean something, they should be generated only from injuries that actually made players unavailable for matches.So I think that the most right thing think would to be remove these useless injuries!

### Removing useless injuries:

In [40]:
injuries_interim_df = injuries_interim_df[injuries_interim_df.games_missed > 0]

In [41]:
injuries_interim_df[injuries_interim_df.games_missed == 0]

,player_id,season_name,injury_reason,from_date,end_date,days_missed,games_missed


### Chacking for duplicates:

I want to ensure that there are not any duplicates within the data, so as duplicates criteria I will define:

Player appearance: **game_id, player_id** \
Lineup: **game_id, player_id** \
Player injury: **player_id, from_date**

So lets check:

In [42]:
appearances_interim_df.duplicated(
    subset=[
        'game_id', 'player_id'
    ]
).any()

np.False_

In [43]:
lineups_interim_df.duplicated(
    subset=[
        'game_id', 'player_id'
    ]
).any()

np.False_

In [44]:
injuries_dupes = injuries_interim_df.duplicated(
    subset=[
        'player_id', 'from_date'
    ],
    keep=False
)

injuries_dupes[injuries_dupes == True].any()

np.True_

In [45]:
injuries_interim_df[injuries_dupes].sort_values(by=['player_id', 'from_date']).head(10)

,player_id,season_name,injury_reason,from_date,end_date,days_missed,games_missed
30240,19447,17/18,Muscle injury,2017-11-25,2017-12-10,16,4
30241,19447,17/18,bruise,2017-11-25,2017-12-04,10,3
35731,21369,18/19,Ankle ligament tear,2019-04-24,2019-05-25,32,5
35732,21369,18/19,Ankle ligament tear,2019-04-24,2019-05-25,32,5
55487,29401,20/21,Calf stiffness,2021-05-07,2021-05-19,13,3
55488,29401,20/21,Calf stiffness,2021-05-07,2021-05-19,13,3
70017,35534,17/18,muscular problems,2017-08-28,2017-09-02,6,1
70018,35534,17/18,muscular problems,2017-08-28,2017-08-31,4,1
83649,41541,20/21,Calf problems,2021-01-14,2021-02-15,33,7
83650,41541,20/21,Calf problems,2021-01-14,2021-02-11,29,6


Now as we can see there are duplicates into the dataset.Also there are exact duplicates which match entierly on each of the columns and there are some which are more specific and which will be the problematic ones!

What I will do is that I will first drop the exact duplicates and then I will think of a way to handle the rest more specific ones:

#### Dropping exact injuries duplicates:
What I have found out is that almost all of the duplicates(**player_id + date**) differ in different way.Some of them only differ in the type of injury.Others differ in the missed days, third in **games missed**.Here is the thing - I don't really care in what they differ as long as they share the same missed games value.Again I will say that for the features we want to create the most important factor is the actual amount of games which the player has missed because of the injury, so if there are duplicates which share the same amount of missed games, but differ in something else, this is not a problem!!!

So lets first drop the exact duplicates:

In [46]:
injuries_interim_df = injuries_interim_df.drop_duplicates()

Now look at the ones which are duplicates and do not differ in the games_missed column:

In [47]:
injuries_interim_df[injuries_interim_df.duplicated(
    subset=['player_id', 'season_name', 'from_date', 'games_missed'], keep=False)]

,player_id,season_name,injury_reason,from_date,end_date,days_missed,games_missed
18591,158863,20/21,Shoulder injury,2021-04-20,2021-04-24,5,1
18592,158863,20/21,Ill,2021-04-20,2021-04-22,3,1
29769,193098,21/22,Muscle injury,2022-01-17,2022-01-20,4,1
29770,193098,21/22,Muscle injury,2022-01-17,2022-01-19,3,1
45697,255755,23/24,Inguinal hernia,2023-12-01,2023-12-18,18,4
45698,255755,23/24,Muscle tear,2023-12-01,2023-12-18,18,4
52810,286384,23/24,Shoulder injury,2023-10-08,2023-10-29,22,3
52811,286384,23/24,Hamstring strain,2023-10-08,2023-10-29,22,3
58240,311287,17/18,Ankle injury,2017-09-26,2017-10-17,22,3
58241,311287,17/18,Ankle sprain,2017-09-26,2017-10-17,22,3


As we can see that these duplicates differ in the end date and in the days missed, but do not differ in the **games_missed** column, which is the most important.So what I will do is that I will first sort the injuries data by player_id and start date to pack the duplicates together and then **descending** by **days_missed**, so that I to take the ones with more days missed, as this is usually the most preferable approach!

#### Sorting the injuries dataset:

In [48]:
injuries_interim_df = injuries_interim_df.sort_values(
    by=['player_id', 'from_date', 'days_missed'], 
    ascending=[True, True, False]
)

Now lets drop the duplicates and leave the ones which are with more days missed:

In [49]:
injuries_interim_df = injuries_interim_df.drop_duplicates(
    subset=[
        'player_id', 'season_name', 'from_date', 'games_missed'
    ],
    keep='first'
)

And now, lastly we should deal with the duplicates which difer in the amount of missed games: 

In [50]:
injuries_interim_df[injuries_interim_df.duplicated(
    subset=[
        'player_id', 'from_date'
    ],
    keep=False
)]

,player_id,season_name,injury_reason,from_date,end_date,days_missed,games_missed
30240,19447,17/18,Muscle injury,2017-11-25,2017-12-10,16,4
30241,19447,17/18,bruise,2017-11-25,2017-12-04,10,3
83649,41541,20/21,Calf problems,2021-01-14,2021-02-15,33,7
83650,41541,20/21,Calf problems,2021-01-14,2021-02-11,29,6
3520,110366,23/24,Cruciate ligament injury,2023-04-17,2023-08-06,112,7
3521,110366,22/23,Meniscus injury,2023-04-17,2023-06-19,64,7
32227,199299,18/19,ankle sprain,2019-02-01,2019-02-22,22,5
32228,199299,18/19,ankle sprain,2019-02-01,2019-02-08,8,1
34029,205503,18/19,Muscle injury,2018-08-24,2018-09-03,11,3
34030,205503,18/19,Muscle injury,2018-08-24,2018-08-31,8,2


What I will do here is that I will again leave the injury which was the longer one.As I do not have enough information and cannot really state what was the case with these players I would say that the possible reason behind these injuries could be the fact that at first, when the player has injured himself, there could have been given a date(like a posible prediction of how long the player will be out), and then due to some circumstances, this date to have been changed!This is what I can think of not, and actually this is a very common and realistic situation in football!

Now lets fix the last remaining duplicates by just leaving the ones with the longer injury:

In [51]:
injuries_interim_df = injuries_interim_df.drop_duplicates(
    subset=[
        'player_id', 'from_date'
    ],
    keep='first'
)

Final validation:

In [52]:
injuries_interim_df.duplicated(
    subset=[
        'player_id', 'from_date'
    ],
    keep=False
).any()

np.False_

In [53]:
appearances_interim_df.shape, lineups_interim_df.shape, injuries_interim_df.shape

((146708, 13), (204082, 10), (11800, 7))

### Exploring the amount of actual matches contained in the datasets:
The appearances and lineups datasets are about playres and their matches.However, I want to see how many matches these datasets cover and are there any inconsistencies in them.This is very important because if these datasets have a lot of missed or inconsistent matches, this will be a very big problem when creating the features!

In order to so I will need to create another table which contains only the unique matches, which means only to drop all of duplicates with criteria: game_id and date.This will leave only one unique match for the date!

In [54]:
test_appearances_data = appearances_interim_df.copy()
test_appearances_data = test_appearances_data.drop_duplicates(subset=[
    'game_id', 'date'
], keep='first')

# Creating a season feature so that I can group by it and see how are the matches distributed across the seasons:
test_appearances_data['season'] = create_season_feature(test_appearances_data, 'date')

In [55]:
games_per_season = test_appearances_data.groupby('season')['game_id'].count().reset_index()
games_per_season.columns = ['season', 'number_of_games']

In [56]:
games_per_season

,season,number_of_games
0,2014/2015,516
1,2015/2016,536
2,2016/2017,525
3,2017/2018,529
4,2018/2019,529
5,2019/2020,512
6,2020/2021,539
7,2021/2022,528
8,2022/2023,514
9,2023/2024,517


Ok for the appearances data it seems that the matches are overall well distributed.And I think that actually, what we see is very true and realistic because if we take the LaLiga matches which are **380** total per season and also add the matches from the other competitions(European cups, domestic cups etc) we will get something like that.

Now lets see the matches specifically from LaLiga:

In [57]:
laliga_games_per_season = test_appearances_data[
    test_appearances_data.competition_id == 'ES1'    
].groupby('season')['game_id'].count().reset_index()

laliga_games_per_season.columns = ['season', 'number_of_games']

In [58]:
laliga_games_per_season

,season,number_of_games
0,2014/2015,380
1,2015/2016,380
2,2016/2017,380
3,2017/2018,380
4,2018/2019,380
5,2019/2020,380
6,2020/2021,380
7,2021/2022,380
8,2022/2023,380
9,2023/2024,380


Ok here everything is perfect.

Now lets check the same thing with the lineups data:

In [59]:
test_lineups_data = lineups_interim_df.copy()
test_lineups_data = test_lineups_data.drop_duplicates(subset=[
    'game_id', 'date'
], keep='first')

test_lineups_data['season'] = create_season_feature(test_lineups_data, 'date')

In [60]:
games_per_season = test_lineups_data.groupby('season')['game_id'].count().reset_index()
games_per_season.columns = ['season', 'number_of_games']

In [61]:
games_per_season

,season,number_of_games
0,2014/2015,534
1,2015/2016,543
2,2016/2017,531
3,2017/2018,541
4,2018/2019,415
5,2019/2020,523
6,2020/2021,551
7,2021/2022,538
8,2022/2023,526
9,2023/2024,532


As we can see, again the matches are overall with realistic counts, but there are also many missing ones, especially in season **2018/2019** when we have **415** matches, which surely is not the right amount of total matches played!

Lets see the LaLiga matches:

In [62]:
target_games = test_appearances_data[test_appearances_data.competition_id == 'ES1'].game_id.values.tolist()

In [63]:
laliga_games_per_season = test_lineups_data[
    test_lineups_data.game_id.isin(target_games)
].groupby('season')['game_id'].count().reset_index()

laliga_games_per_season.columns = ['season', 'number_of_games']

In [64]:
laliga_games_per_season

,season,number_of_games
0,2014/2015,380
1,2015/2016,380
2,2016/2017,380
3,2017/2018,380
4,2018/2019,332
5,2019/2020,380
6,2020/2021,380
7,2021/2022,379
8,2022/2023,380
9,2023/2024,380


As I have expected, there are missing matches in season **2018/2019**, and there are 48 missing matches, which is not good!This would mean that for these matches certain players will be with unidentified starting info(did he start the match or he was a substitute).This is important info which we just cannot know at this moment with this data!However, I will keep this is mind and when we create the feature we should think what to do with these missing players!

In [65]:
del test_appearances_data, test_lineups_data

### This was the cleaning and the exploration part of this notebook.Now lets proceed with the most important work:

---

## Feature Engineering:
Lets start with the creation of the `player influence, match importance and team presure` features.

As these three features are different and require different processing, I will seperate the work and start with the creation of the player influence features.One big factor in the player influence features is exactly the players availability in the matches and this is the reason why I have spended so much time in cleaning and preperation!

#### Player importance and the time factor in the football matches:
The most important question we should aks is: What defines a player influence and his importance during the matches?Something more, in what way this importance will be collected and what should be the time period in which the features will be created?

On the question what defines the player influence, there are really many factors which should be considered.For example how much time he has played throughout the matches and more importantly, what was his contribution for the teams in these matches!By contribution I many things like goals, assists and even team strength without the player!We should consider the how the team is working without the availability of a certain player.**The answers of these questions define the importance of a player and his influence for the matches!**

Now, something else we should think about: The football matches as the main data of out project and their outcomes as the target variable which we want to predict, **highly depend on the time as a factor!**.Why?In order to create a model which can really capture the underlying trends and features which influence the matches, we should consider **the time** as a factor from which we will create most of the features for the models.By using the time factor, we can create features such as **rolling averages** for example!By rolling averages I mean teams statistics like goals, assists, ball possesion, points gathered, etc., in a predefined **time window**!When we create such time-dependent features, we give, to our models, the oppotunity to observe the team's performance and their strenght at a bigger level!And we will think about this for a moment, actually, this is the only thing we can give to the predictive models as a useful info about the teams and their **likelihood** of winning the match.If we don't give this, we leave our models with very basic and static data from which nothing can be extracted!

The purpose of these explanations, is to emphasize the **importance** of the time in the features that we will creating and to say that in order to create really useful and meaningful features, we should not only use what we have at the moment of the matches, because most of the time it is nothing important, but to look at a bigger level and idenity the true, underlying factors which can define the outcomes of the matches!

#### Time as a Factor (Recency Weighting)
Time acts as a critical factor because **recent form** is much more relevant than form from two years ago. Many sports models account for this by utilizing an **Exponential Moving Average (EMA)**, which naturally gives heavier weight to a team's most recent matches and decays the impact of older games.**This is something really important and it should be considered when difining the time window in which the features will be created!**

Now lets start with the creation of the **historical player-strength** table.This will be the base from which the player importances will be defined!

#### Creating the historical player-strength table:
Before starting anything, we should define the Time Window as the date range from which the player importance features will be calculated.As I said, the **recent form**, of a player or a team at all, is much more important and relevant than a from really past form!And because of that I will define consider the last **10** games of the players from which their importance will be defined!As we will be working with the **appearances** data, 10 games refer to 10 appearances of a player or 10 games of that player!

In [66]:
WINDOW

10

#### Sorting the appearances table:

In [67]:
appearances_interim_df = appearances_interim_df.sort_values(
    ["player_id", "date"]
)

### Creating starter flags
One of the components which will define the player importance feature will be the time that the player has played during the matches or the start share which he has!It is important to identity if a player is a starter or a substitute!

We can see this using the players **lineups data**:

In [68]:
lineups_interim_df.type.value_counts()

type
starting_lineup    112973
substitutes         91109
Name: count, dtype: int64

In [69]:
lineups_interim_df['is_starter'] = (
    lineups_interim_df["type"]
    .str.lower()
    .str.contains("starting")
    .astype(int)
)

#### Creating player match starter table:
This table will contain the most important info that we should know about the players from their teams lineups!

In [70]:
starter_info = lineups_interim_df[
    [
        "game_id",
        "player_id",
        "club_id",
        "position",
        "team_captain",
        "is_starter"
    ]
].copy()

### Merging lineups with appearances
Now I will merge the created starter_info table with the appearances data:

In [71]:
player_matches = appearances_interim_df.merge(
    starter_info,
    on=["game_id", "player_id"],
    how="left"
)

In [72]:
player_matches.info()

<class 'pandas.DataFrame'>
RangeIndex: 146708 entries, 0 to 146707
Data columns (total 17 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   appearance_id           146708 non-null  str           
 1   game_id                 146708 non-null  int32         
 2   player_id               146708 non-null  int32         
 3   player_club_id          146708 non-null  int16         
 4   player_current_club_id  146708 non-null  int32         
 5   date                    146708 non-null  datetime64[us]
 6   player_name             146708 non-null  category      
 7   competition_id          146708 non-null  category      
 8   yellow_cards            146708 non-null  int8          
 9   red_cards               146708 non-null  int8          
 10  goals                   146708 non-null  int8          
 11  assists                 146708 non-null  int8          
 12  minutes_played          146708 non-null  

There are around 2 thousand missing values.As it seems, there are many players which vary across the two datasets(appearances and lineups)!Of course this is a cause from the thing we discovered in the exploration part, and it is that for season 2018/2019 there are around 150 missing games!And specifically for the LaLiga competition there are 48 missing games in that season!For sure, this is not the only problem which causes this big amount of missing players!

Lets observe what is happening:

In [73]:
missing_games = (
    player_matches.loc[
        player_matches["position"].isna(),
        "game_id"
    ]
    .unique()
)

len(missing_games)

119

In [74]:
missing_rows = player_matches[
    player_matches["position"].isna()
]

len(missing_rows)

2710

In [75]:
missing_rows["player_id"].nunique()

516

There are **119** missing games, **516** missing players and **2710** missing rows at all.

Now before making any hasty decisions, lets check how much these missing values affect the dataset:

In [76]:
affected_match_pct = (
    len(missing_games)
    /
    appearances_interim_df["game_id"].nunique()
)

affected_match_pct

0.020616770616770617

The affection seems to be quite small!One possible approach is to remove these missing games.And this would be the most professional way to handle this data, because if I do not remove them, I need to fill them up with some values - what values should I put for players which I just don't know!And these values should be about: have they started the match or not, their position etc.All of these values play a very big role into the player importance features and just to put some naive and unrealistic values would be very bad mistake!On top of that I have observed all of the datasets, their cleanings, filtering and everything else which could be related to these missing matches, and all of them lead to the single explanation - There are just missing matches in the Transfermarkt lineups dataset and that's it!!!

Now before removing the values lets check, if they are actually missing from the LaLiga competition or from the other(more unimportant) european competitions(Champtions league, League Europe, Copa Del Ray etc):

In [77]:
matches_interim_df[
    matches_interim_df["game_id"].isin(missing_games)
]["competition_id"].value_counts()

competition_id
CDR     68
ES1     49
CL       1
EL       1
CLQ      0
ECLQ     0
ELQ      0
EURO     0
FIWC     0
KLUB     0
SUC      0
UCOL     0
USC      0
Name: count, dtype: int64

Well, as we can see that most of them are from **Copa Del Ray**, but there is also a **significant** number from **LaLiga**.However, **not significant enough** to leave them into the dataset!We should really answer the question: What would be bad: to have these matches in the dataset, but with invalid values, or to not have them, but without invalid values which could make a bias to the model or something like that?I would go with the second approach!So lets remove these values?

#### Removing matches with missing data: 

In [78]:
bad_games = missing_rows["game_id"].unique()

Removing the invalid matches from the appearances dataset:

In [79]:
appearances_interim_df = appearances_interim_df[
    ~appearances_interim_df["game_id"].isin(bad_games)
]

Removing the invalid matches from the lineups dataset:

In [80]:
lineups_interim_df = lineups_interim_df[
    ~lineups_interim_df["game_id"].isin(bad_games)
]

In [81]:
appearances_interim_df.shape, lineups_interim_df.shape

((143970, 13), (204046, 11))

Merging again:

In [82]:
starter_info = lineups_interim_df[
    [
        "game_id",
        "player_id",
        "club_id",
        "position",
        "team_captain",
        "is_starter"
    ]
].copy()

In [83]:
player_matches = appearances_interim_df.merge(
    starter_info,
    on=["game_id", "player_id"],
    how="left"
)

In [84]:
player_matches.isna().any().any()

np.False_

In [85]:
player_matches.shape

(143970, 17)

As we can see, there are no longer missing values.However, what I will do is that I will save the id's of the invalid matches so that later I can preprocess them more detaily and eventually fix the problem!

In [86]:
bad_games_df = pd.DataFrame({
    "game_id": bad_games
})

In [87]:
PROJECT_ROOT = Path().resolve().parent.parent
INTERNAL_DATA_PATH = 'data/internal/'

file_path = PROJECT_ROOT / INTERNAL_DATA_PATH
file_path.mkdir(parents=True, exist_ok=True)

In [88]:
file_path = PROJECT_ROOT / INTERNAL_DATA_PATH / 'matches_without_lineups_info.csv'
try:
    bad_games_df.to_csv(file_path, index=False, mode='x')
except FileExistsError:
    print(f"File already exists at {file_path}. Skipping save.")

File already exists at C:\Users\Asus\Desktop\Data_Science\Football_Betting_Analysis\Football_Betting_Analysis\data\internal\matches_without_lineups_info.csv. Skipping save.


---

Now lets proceed with the creation of the features:

### Creating rolling players metrics:
Now it's time to create the rolling features from which the player importance score will be defined.In order to avoids leakage, I will shift by 1 match!

For the features I have implemented several functions which in different ways calculate the rolling metrics.The first and most standard one is the `calculate_rolling_metric`, which will create a **rolling metric** based on a provided function(e.g. sum, mean).I called this function the standard one, because it calculates the rolling features using a predefined time window(e.g 5, 10) and it takes all of the values up until that window and includes them into the calculations of the rolling feature.However, the thing is that this function gives equall weights to all of the values, without making difference between the more recent matches the more older ones!  

However, I have also implemented the `calculate_ewma_average` function which calculates *Exponentially Weighted Moving Average*.The purpose of this function is to give more account to the more recent matches by putting exponentially descresing weights to the more older matches.Something more, this function includes all of the past matches and it controls their weights using the **span** and adjust parameters.

The other function is the `calculate_time_based_ewma` function which in contrast to the `calculate_ewma_average` function, it calculates the rolling metric using the time-based EWMA instead of row-index EWMA.The functions allows you to pass a datetime index using the **times** parameter.As a result this will exponentially decays a player's form based on actual days passed, meaning their form naturally rusts over long periods of absence.So this function is actually very useful because it captures the player's absence and it rightfully accounts that their form might had have become weaker, as it is most of the times!

#### About the parameters **span** and **adjust**:
THese two paramteres are the essential driven factors of the ewma functions because from them depends how the values of the different data points contribute to the calculation of the rolling feature!

##### How they work mathematically:
The `span` controls the smoothing factor $(\alpha)$(alpha).The `calculate_ewma_average` function automatically calculates it using this exact formula:
$$
    \alpha = \frac{2}{span + 1}
$$
This means that if span is set to 5, alpha becomes $\frac{2}{6} = 0.33$. This means the very last match accounts for **33.3%** of the entire moving average value, while the remaining **66.7%** comes from all prior historical games combined.

The `adjust` parameter changes how pandas initializes the moving average when there is very little history (e.g., games 1, 2, and 3 of the season)
- adjust=False (Recursive Mode): It assumes the very first game is perfectly representative of the team's baseline ability. It initializes the calculation by setting the first average equal to the first game's score ($y_0 = x_0$)
- adjust=True (Weighted Average Mode): It treats the early games as limited samples. It dynamically scales down the importance of that first match so a single extreme blowout in Game 1 doesn't warp the team's metrics for the next 5 games

**Why I would use the adjust parameter set to False**:
- The data which we are working with is time series data which is grouped by seasons, so setting `adjust=False` allows the previous match's result to be recursively scaled with a weight of $\alpha$ without altering the starting values. It is highly useful in sports analytics for establishing a "prior" (like the team's average attacking strength) and iteratively updating it for each match as the season progresses!

The implementations of the functions: [features_creation](../../src/football_betting_analysis/features/features_creation.py)

Now I will start with the creation of the rolling minutes features.For this feature I will use the `calculate_ewma_average`, as I want give more account to the more recent matches!Firstly I will need to create a normalize the minutes into percantages so that after that the EWMA is directly operating on a normalized variable! 

In [89]:
player_matches["minutes_pct"] = (
    player_matches["minutes_played"] / 90
)

player_matches["minutes_share"] = calculate_ewma_average(
    data=player_matches,
    group_col="player_id",
    column="minutes_pct",
    span=5,
    adjust=False
)

Now lets calculate the **rolling starts** of the players.Again, I will use the `calculate_ewma_average` because starting status changes rapidly.Managers rotate.Transfers happen.Injuries happen.And because of that, I would put more weight on the more recent matches than giving the whole values equal weights!

In [90]:
player_matches["start_share"] = calculate_ewma_average(
    data=player_matches,
    group_col="player_id",
    column="is_starter",
    span=5,
    adjust=False
)

Now I will proceed with captain frequency feature which will define the consistency of a player to be the captain of the team, which of course will contribute a lot his importance in the matches and for the team at all!

First lets create a captain flag, which will indicate if the player has been the captain of the specific game and then I will create a captain frequency rolling average.This time I will use the `calculate_rolling_metric` function, as being a captain is overall a stable indicator, so it will be enough to see the player's captaincy in their previous 10 games!

In [91]:
player_matches["captain_flag"] = (
    player_matches["team_captain"]
    .fillna(False)
    .astype(int)
)

In [92]:
player_matches["captain_frequency"] = calculate_rolling_metric(
    data=player_matches,
    group_col='player_id',
    column='captain_flag',
    func='mean',
    window=WINDOW
)

Now lets proceed with the more important features.By important I mean the ones which can easily identify the player's influence in the matches.The **goals and assist**!For these two features I will use the `calculate_rolling_metric` function because for these specific features I much more interested to get the raw counts as goals for the players in the last 10 matches.Also after I create these features I will also need to create a `goals_per90` and `assists_per90` features which represent how the players perform within the game's 90 minutes interval - This is really important because if we want to capture the real influence of the players in the matches, we should make our calculations to be into the range of the matches themselves!

I will start with the **goals**:

In [93]:
player_matches["rolling_goals"] = calculate_rolling_metric(
    data=player_matches,
    group_col='player_id',
    column='goals',
    func='sum',
    window=WINDOW
)

Now the same for the **assists**:

In [94]:
player_matches["rolling_assists"] = calculate_rolling_metric(
    data=player_matches,
    group_col='player_id',
    column='assists',
    func='sum',
    window=WINDOW
)

Now before creating the `goals_per90 and assists_per90` features I will need to first create a rolling minutes feature in the range of the 10 last matches!Then I will use the **rolling minutes** to divide them from the rolling goals and assists and multiply the result by 90 to get the `goals_per90 and assists_per90` features!

In [95]:
player_matches["rolling_minutes"] = calculate_rolling_metric(
    data=player_matches,
    group_col='player_id',
    column='minutes_played',
    func='sum',
    window=WINDOW
)

And now I can create the per90 features:

In [96]:
player_matches["goals_per90"] = (
    player_matches["rolling_goals"]
    /
    player_matches["rolling_minutes"]
    * 90
)

player_matches["assists_per90"] = (
    player_matches["rolling_assists"]
    /
    player_matches["rolling_minutes"]
    * 90
)

Now with these two features(`goals_per90 and assists_per90`) I will define the players contribution score.And of course before doing that I'll need to fill any missing values of these two columns!

#### About the missing values:
With the creation of all these rolling features, the dataset currently has a lot of missing values and this because of the reason that for these rolling features to be created, there should be values before the current matches, and if there are not at least 1 value to take for the calculation(based on the **min_periods** param), the functions will return **NaN**.And as expected for the first matches of the players, there are no past values to take so all of the rolling features will be with NaN values.And in order for me to create some features I am required to fill this missing values with something.In the current situation the most logical value would be **0**! 

In [97]:
player_matches[
    ["goals_per90", "assists_per90"]
] = player_matches[
    ["goals_per90", "assists_per90"]
].fillna(0)

Now lets create the player's contribution score.

And of course I know that this **contribution score** will not be **relevant** to all of the different players in the dataset because they are with different positions and purposes at all.For example how would a contribution score defined from goals and assists will be relevant to a goalkeeper or to a defender?So for now I will just create the feature like that, and then when defining the **importance score** of the different players I will take into account the different types of players!

In [98]:
player_matches["contribution_score"] = (
    player_matches["goals_per90"]
    +
    0.7 * player_matches["assists_per90"]
)

the final list of features:

In [99]:
player_matches.columns

Index(['appearance_id', 'game_id', 'player_id', 'player_club_id',
       'player_current_club_id', 'date', 'player_name', 'competition_id',
       'yellow_cards', 'red_cards', 'goals', 'assists', 'minutes_played',
       'club_id', 'position', 'team_captain', 'is_starter', 'minutes_pct',
       'minutes_share', 'start_share', 'captain_flag', 'captain_frequency',
       'rolling_goals', 'rolling_assists', 'rolling_minutes', 'goals_per90',
       'assists_per90', 'contribution_score'],
      dtype='str')

### Now as the main features have been created, I should proceed by separating the players into their positional groups

This is extremely important, because each of the players has a different purpose in the matches and we can't lump them all together, because they are just different.And in order to create a reliable and realistic player's importance score, the separation of the players is just mandatory!

So first lets see what type of player's position do we have in the dataset:

In [100]:
player_matches.position.unique().tolist()

['Centre-Forward',
 'Centre-Back',
 'Left Midfield',
 'Attacking Midfield',
 'Left Winger',
 'Goalkeeper',
 'Second Striker',
 'Left-Back',
 'Central Midfield',
 'Defensive Midfield',
 'Right-Back',
 'Right Winger',
 'Right Midfield',
 'midfield',
 'Sweeper']

As we can see from the positions, we can distribute them into the following categories: 
- **Midfielders**: which refers to the playmakers in the game(The ones which play in the center of the pitch)
- **Defenders**: which refers to the defensive players which role is to stop the attackers of the opposite team
- **Attackers**: which refers to the top of the pitch and ofthen act as the star players, which scores goals and decide the matches!

If you don't understand the `Sweeper` position, it mainly refers to a specialized, deep-lying defensive player. Positioned just behind the main defensive line, the sweeper has no strict man-marking duties. Instead, they "sweep up" or mop up any loose balls, intercept through-balls, and serve as a final safety net!So yes it falls into the group of **Defenders**!

So I will create a function which will assign a group to each of the players by looking at their possitions:

In [101]:
def map_position(pos):
    pos = str(pos).lower()

    if any(x in pos for x in ['forward', 'winger', 'striker']):
        return "FWD"

    if any(x in pos for x in ['back', 'sweeper']):
        return "DEF"

    if any(x in pos for x in ['midfield']):
        return "MID"

    return "GK"

Now lets apply the function to the players:

In [102]:
player_matches["position_group"] = (
    player_matches["position"]
    .apply(map_position)
)

In [103]:
player_matches.position_group.value_counts(dropna=False)

position_group
DEF    46028
FWD    45792
MID    42208
GK      9942
Name: count, dtype: int64

The distribution of the different player's positions is actually very good, because this means that there is data for all types of players in the dataset!And of course the goalkeepers are less just because the goalkeepers of a team is moct of times only one player, and possibly two max if the first goalkeepers gets injured during a match!

Now before proceeding ahead, there is something very important that I have missed to do.Currenly the rolling fatures are scaled differently and this is because I have used different rolling function for their calculations.For the **minutes_share and start_share** I have used the `calculate_ewma_average` function which scales the data into a range of **[0 - 1]**.However, for the **captain_frequency and contribution_score** I have used the `calculate_rolling_metric` function which can produce values larger than 1.This will not be a very big problem when calculating the importance score, but still it is possible to get some strange results!So for this reason, I will need to normalize the contribution score feature so that it scales its values in the range between 0 and 1!The captain_frequency, actually, does not need to be scaled because it used the captain_flag which is a binary variable and then gets the mean from the values, which guarantees us that the values of this feature cannot exceed 1!

So lets make the normalization!
### Normalizing the cintribution score feature:
For the normalization, I will perform a bounded scaling using the `np.tanh` function which calculates the hyperbolic tangent of the values. It maps any real-valued number to a continuous output strictly between **[-1 - 1]**.However, the function will not produce any negative values, just because the contribution score as a feature does not contain values smaller than 0, so this means that the values will be in the range **[1 - 1]**

What I am trying to achive with this scalling is to not allow the contribution score to dominate the players importance score.And this will happen if I don't scale the feature because there are values larger than 1, up to 2 which can highly affect the importance score of the playres.So I should ensure that all of the features share one range of values!

In [104]:
player_matches["contribution_score_norm"] = np.tanh(
    player_matches["contribution_score"]
)

Now as everything is normalized, grouped and scaled correctly, lets created the player's importance score feature.For the importance score I will again create a function which based on the group of the player will apply different values to the different rolling features(that we have created) and this way it will form the player importance scores using all of the different components(the rolling features that we have created.Actually it will use only those which are relevant to the position group of the player - for example, into the importance score of the goalkeeper I will not include the **contribution_score** feature, **because a goalkeeper has nothing to do with goals and assists!**)

### Creating the importance score function:
Now the way that this function wil work, is that it will combine each of the features that we have created up until this moment(`minutes_share, start_share, captain_frequency, conyribution_score`) and for each of these features it will apply different weights based on the position of the player.So as I have already said, not all of the feature will be used into the calculations of their scores, because some of the features are not relevant to the positions and types of players.This will ensure that the scores of the playres and their importances will be cretaed, considering only the components and feature which are related to them!The weighs which are used in the function are created in way that mostly corresponds with the feature(with which the weight is used) and also the weights are created so that for the most important game components of the players, to have the strongets weights.So for example the weight of the contribution score feature for a striker will be much more bigger than the contribution score weight for the defenders!

In [105]:
def compute_importance(row):

    minutes_share = row["minutes_share"]
    start_share = row["start_share"]
    captain_frequency = row["captain_frequency"]
    contribution_score = row["contribution_score_norm"]

    p = row["position_group"]

    if p == "GK":
        return (
            0.60*minutes_share +
            0.30*start_share +
            0.10*captain_frequency
        )

    elif p == "DEF":
        return (
            0.50*minutes_share +
            0.35*start_share +
            0.15*captain_frequency
        )

    elif p == "MID":
        return (
            0.40*minutes_share +
            0.25*start_share +
            0.15*captain_frequency +
            0.20*contribution_score
        )

    else:
        return (
            0.40*minutes_share +
            0.25*start_share +
            0.10*captain_frequency +
            0.25*contribution_score
        )

Now lets finally apply the function:

In [106]:
player_matches["importance_raw"] = (
    player_matches
    .apply(compute_importance, axis=1)
)

In [107]:
player_matches.columns

Index(['appearance_id', 'game_id', 'player_id', 'player_club_id',
       'player_current_club_id', 'date', 'player_name', 'competition_id',
       'yellow_cards', 'red_cards', 'goals', 'assists', 'minutes_played',
       'club_id', 'position', 'team_captain', 'is_starter', 'minutes_pct',
       'minutes_share', 'start_share', 'captain_flag', 'captain_frequency',
       'rolling_goals', 'rolling_assists', 'rolling_minutes', 'goals_per90',
       'assists_per90', 'contribution_score', 'position_group',
       'contribution_score_norm', 'importance_raw'],
      dtype='str')

Ok now as the importance score feature has been created, we need to go even further by normalizing this feature into a **team relative strength**.What I mean by this is that I will create a **total team importance score**, by summing all of the players importance's scores!After I create the teams totals, I will create the final player importance score function as a result from the division between the player importance_raw(the initial player importance score feature that we created above) and the teams totals.This feature will really represent the player influence because the value of the feature will be calculated relative to the sum of all other player's importance scores!

First lets create the team totals:

In [108]:
team_totals = (
    player_matches
    .groupby(
        [
            "player_club_id",
            "date"
        ]
    )["importance_raw"]
    .transform("sum")
)

Now lets create the final player importance score feature, this time calculated relative to the team totals!

### Creating the final player importance score feature:

In [109]:
player_matches["importance_score"] = (
    player_matches["importance_raw"]
    /
    team_totals
)

In [110]:
player_matches["importance_score"] = (
    player_matches["importance_score"]
    .fillna(0)
)

Ok now as we have defined, **what a player importance score is**, its time to proceed with the creation of the other features.If you remember the introduction of the notebook, I have specified and explained some of the aim features that this notebook should create and here they are:
- missing_players_count
- missing_key_players_count
- missing_star_players_count
- missing_importance_sum - The sum of the missing players importance score
- missing_importance_ratio - The division of the missing players importance score with the total team squad importance
- missing_defenders
- missing_midfielders
- missing_forwards
- starting_goalkeeper_missing

As you can see, all of the features are about missing players.However, I will not consider the unavailability of a player the only factor which defines a player influence, so I can also create a feature about the available star players of a team or something like that.

Also something else you can see.In order for us to identify if a player is a star, key or whatever, first we should have had created the player importance score, with which now we can say if a player is key, star etc.So if you are following my point, I want to say that everything which we have created is for a reason and nothing is useless!

Now lets actually proceed with the creation of the rest features:
### Team ranking, key players, star players:
In order for me to categorize the contribution of the players,I will need to create a team ranking: 

In [111]:
player_matches["team_rank"] = (
    player_matches
    .groupby(
        [
            "player_club_id",
            "date"
        ]
    )["importance_score"]
    .rank(
        ascending=False,
        method="dense"
    )
)

The code above groups the players into team and matches dates, so that it can ranks them for their club and the specific match date.The **ascending=False** means that the highest score gets rank 1, the second-highest gets rank 2, and so on.**method="dense"** will handle ties without skipping numbers. If two players tie for rank 2, the next player will strictly be rank 3 (instead of skipping to rank 4).

Now lets identify the star players:
#### Identifying the star players:

In [112]:
player_matches["is_star_player"] = (
    (player_matches["importance_score"] > 0) &
    (player_matches["team_rank"] <= 3)
).astype(int)

Now lets define what a key player means:

In [113]:
player_matches["importance_pct"] = (
    player_matches
    .groupby(
        [
            "player_club_id",
            "date"
        ]
    )["importance_score"]
    .rank(pct=True) # This time we rank the players as percantages of the whole group(The player's team)
)

In [114]:
player_matches["is_key_player"] = (
    player_matches["importance_pct"] >= 0.75 # The player must have importance score higher than or equal to 75 percent of his team's players scores!
).astype(int)

In [115]:
player_matches.columns

Index(['appearance_id', 'game_id', 'player_id', 'player_club_id',
       'player_current_club_id', 'date', 'player_name', 'competition_id',
       'yellow_cards', 'red_cards', 'goals', 'assists', 'minutes_played',
       'club_id', 'position', 'team_captain', 'is_starter', 'minutes_pct',
       'minutes_share', 'start_share', 'captain_flag', 'captain_frequency',
       'rolling_goals', 'rolling_assists', 'rolling_minutes', 'goals_per90',
       'assists_per90', 'contribution_score', 'position_group',
       'contribution_score_norm', 'importance_raw', 'importance_score',
       'team_rank', 'is_star_player', 'importance_pct', 'is_key_player'],
      dtype='str')

Now with these two features we can understand much more about the players and their influence over the matches! \
Now lets proceed with the creation of all these features:
- missing_players_count
- missing_key_players_count
- missing_star_players_count
- missing_importance_sum
- missing_importance_ratio
- missing_defenders
- missing_midfielders
- missing_forwards
- starting_goalkeeper_missing
> These features have been explained at the begining of the notebook!

Here are some other features which I also plan to include:
- missing_captain
- missing_top1_player - I will get these player by looking at the highest player importance score plus the player ranking(which should be strongly **1**)
- available_strength - The total importance score of all of the players which are available for the specific match
- missing_expected_starter_strength
- missing_gk_strength - missing importance score of the goal keepers
- missing_def_strength - missing importance score of the defenders
- missing_mid_strength - missing importance score of the midfielders
- missing_fwd_strength - missing importance score of the strikers
- squad_stability

In order to create these features we need to clear some things out.What is our goal: \
For every match we have a teams which are playing one against anoter and a match date right! \
We want to know:

**Which players belong to these teams immediately before match date?** \
**Which of them are injured on the match date?** \
**How important are they?**

Everything else comes from these answers!

Now lets begin. \
First thing I will do is to create the `expected_starter` feature, which will identify if the player is expected to start the game in the starting eleven!I will define these feature using the `start_share` feature that we have created a bit earlier.I need to create this feature because later it will be used in many places(e.g to identify if the missing goalkeeper of a team is the usual starting goalkeeper at all or mayble it is his substitute?):

### Creating the expected_starter feature:

In [116]:
player_matches["expected_starter"] = (
    player_matches["start_share"] >= 0.60
).astype(int)

Now before proceeding with the other feature I will create a **player state** table which will just take the most important features that we have created about the players from the `player_matches` table.Later this table will be merged with the actual matches!

### Creating a player state table:

In [117]:
player_state = player_matches[
    [
        "player_id",
        "player_club_id",
        "game_id",
        "date",
        "importance_score",
        "is_star_player",
        "is_key_player",
        "position_group",
        "captain_flag",
        "minutes_share",
        "start_share",
        "captain_frequency",
        "expected_starter"
    ]
].copy()

In [118]:
player_state.columns

Index(['player_id', 'player_club_id', 'game_id', 'date', 'importance_score',
       'is_star_player', 'is_key_player', 'position_group', 'captain_flag',
       'minutes_share', 'start_share', 'captain_frequency',
       'expected_starter'],
      dtype='str')